## Import Packages

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.optim import Adam
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.preprocessing import StandardScaler                                               
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score                                                         
from pathlib import Path
from joblib import dump

## Import Datasets

In [10]:
test_data_X = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/test-data-senegal-input-t0.csv')
test_data_y_t1 = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/test-data-senegal-output-map-t1.csv', header=None)
test_data_y_t0 = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/test-data-senegal-output-map-t0.csv', header=None)

In [11]:
train_data_X = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/train-data-senegal-input-t0.csv')
train_data_y_t1 = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/train-data-senegal-output-map-t1.csv', header=None)
train_data_y_t0 = pd.read_csv('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/train-data-senegal-output-map-t0.csv', header=None)

In [12]:
resolution_y, resolution_x = 64, 64

test_data_y_t1 = test_data_y_t1.to_numpy().reshape(len(test_data_y_t1), resolution_y, resolution_x)
train_data_y_t1 = train_data_y_t1.to_numpy().reshape(len(train_data_y_t1), resolution_y, resolution_x)

In [13]:
fract_storm_test = np.sum(test_data_y_t1) / len(test_data_y_t1.flatten())
fract_storm_train = np.sum(train_data_y_t1) / len(train_data_y_t1.flatten())

print(fract_storm_test)
print(fract_storm_train)

0.007722339563414307
0.00792849799735096


### Preprocessing

Log transforming

In [14]:
train_data_X, val_data_X, train_y_lt1, val_y_lt1 = train_test_split(train_data_X, train_data_y_t1, test_size=0.3, random_state=12)

In [ ]:
for col in ['size1', 'size2', 'size3', 'wp1', 'wp2', 'wp3', 'd1', 'd2', 'd3']:
    train_data_X.loc[:, col] = np.log1p(train_data_X[col].astype(float))
    test_data_X.loc[:, col] = np.log1p(test_data_X[col].astype(float))
    val_data_X.loc[:, col] = np.log1p(val_data_X[col].astype(float))

/tmp/ipykernel_967812/3849798946.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[6.7833252  9.07050343 8.05547514 ... 6.24222327 5.0369526  6.07073773]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_data_X.loc[:, col] = np.log1p(train_data_X[col])
/tmp/ipykernel_967812/3849798946.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[8.40603814 8.19891444 6.38856141 ... 7.7151236  8.08917568 8.29779263]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  test_data_X.loc[:, col] = np.log1p(test_data_X[col])
/tmp/ipykernel_967812/3849798946.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[6.77308038 7.74283596 5.33753808 ... 5.14749448 6.7310181  5.33753808]' has dtype incompatible w

### Scaling

In [16]:
# Exclude mask columns from scaling
mask_cols = ['mask1', 'mask2', 'mask3']

# Final list of features to scale
cols_to_scale = [col for col in train_data_X.columns 
                 if col not in mask_cols]

scaler = StandardScaler()

X_train_scaled = train_data_X.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(train_data_X[cols_to_scale])

X_val_scaled = val_data_X.copy()
X_val_scaled[cols_to_scale] = scaler.transform(val_data_X[cols_to_scale])

X_test_scaled = test_data_X.copy()
X_test_scaled[cols_to_scale] = scaler.transform(test_data_X[cols_to_scale])

dump(scaler, "/content/drive/MyDrive/Zambia/t0-only/Zambia-scaler-lt1-t0-only.bin", compress=True)

In [19]:
x_train = X_train_scaled
y_train = train_y_lt1

x_val = X_val_scaled
y_val = val_y_lt1

x_test = X_test_scaled
y_test = test_data_y_t1

In [21]:

class UpsampleCNN(nn.Module):
    def __init__(self, input_dim=23, initial_size=8, output_size=(64, 64)):
        super(UpsampleCNN, self).__init__()
        self.initial_size = initial_size

        self.fc1 = nn.Linear(input_dim, 256)
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(256, initial_size * initial_size * 16)

        self.conv1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.final_conv = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = x.view(-1, 16, self.initial_size, self.initial_size)  # Reshape to (B, C, H, W)

        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = F.relu(self.conv1(x))

        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = F.relu(self.conv2(x))

        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = F.relu(self.conv3(x))

        x = torch.sigmoid(self.final_conv(x))  # Final output in range [0, 1]
        return x

# Create model instance
model = UpsampleCNN()
print(model)


UpsampleCNN(
  (fc1): Linear(in_features=23, out_features=256, bias=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=256, out_features=1024, bias=True)
  (conv1): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (final_conv): Conv2d(16, 1, kernel_size=(1, 1), stride=(1, 1))
)


In [ ]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryAUROC, BinaryAveragePrecision, BinaryF1Score

threshold = 0.15

precision = BinaryPrecision(threshold=threshold)
recall = BinaryRecall(threshold=threshold)
f1_score = BinaryF1Score(threshold=threshold)
auc = BinaryAUROC()
prc = BinaryAveragePrecision()

In [31]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [25]:
class LightningUpsampleModel(pl.LightningModule):
    def __init__(self, input_dim=23, lr=1e-3, threshold=0.15):
        super().__init__()
        self.model = UpsampleCNN(input_dim=input_dim)  # use the model from earlier

        self.criterion = nn.BCELoss()

        # Metrics
        self.precision = BinaryPrecision(threshold=threshold)
        self.recall = BinaryRecall(threshold=threshold)
        self.f1 = BinaryF1Score(threshold=threshold)
        self.auc = BinaryAUROC()
        self.prc = BinaryAveragePrecision()
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze(1)
        loss = self.criterion(y_hat, y)

        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze(1)

        self.precision.update(y_hat, y)
        self.recall.update(y_hat, y)
        self.f1.update(y_hat, y)
        self.auc.update(y_hat, y)
        self.prc.update(y_hat, y)

        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss, prog_bar=True)

    def on_validation_epoch_end(self):
        self.log('val_precision', self.precision.compute(), prog_bar=True)
        self.log('val_recall', self.recall.compute(), prog_bar=True)
        self.log('val_f1', self.f1.compute(), prog_bar=True)
        self.log('val_auc', self.auc.compute(), prog_bar=True)
        self.log('val_prc', self.prc.compute(), prog_bar=True)

        self.precision.reset()
        self.recall.reset()
        self.f1.reset()
        self.auc.reset()
        self.prc.reset()

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=self.lr, clip_value=1.0)


In [26]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Define early stopping (already done earlier)
early_stop_callback = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    verbose=True
)

# Optional: Save the best model
checkpoint_callback = ModelCheckpoint(
    monitor="val_auc",
    mode="max",
    save_top_k=1,
    verbose=True,
    filename="best-model"
)


In [30]:
# Create the trainer
trainer = Trainer(
    devices=4,
    max_epochs=20,
    callbacks=[early_stop_callback, checkpoint_callback],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [31]:
model.eval()
with torch.no_grad():
    pred_y_train = model(torch.from_numpy(x_train).float().to(model.device)).cpu().numpy().flatten()
    pred_y_test = model(torch.from_numpy(x_test).float().to(model.device)).cpu().numpy().flatten()

TypeError: expected np.ndarray (got DataFrame)

In [ ]:
def reliability_curve(y_true, y_pred, bin_size=0.1, min_predictions_per_bin=50):
    """
    Computes the reliability curve (calibration curve) for a binary classifier.
    
    Parameters:
        y_true (array-like): Ground truth binary labels (0 or 1).
        y_pred (array-like): Predicted probabilities (between 0 and 1).
        bin_size (float): Width of the bins to divide probability space.
        min_predictions_per_bin (int): Minimum number of predictions in a bin to be included.
        
    Returns:
        bin_centers (list): Midpoints of the bins used.
        bin_positive_rates (list): Observed frequency of positive class in each bin.
        bin_counts (list): Number of predictions in each bin.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    bins = np.arange(0, 1 + bin_size, bin_size)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    bin_positive_rates = []
    bin_centers_output = []
    bin_counts = []

    for lower, upper, center in zip(bins[:-1], bins[1:], bin_centers):
        in_bin = (y_pred >= lower) & (y_pred < upper)
        count_in_bin = np.sum(in_bin)
        
        if count_in_bin >= min_predictions_per_bin:
            observed_rate = np.mean(y_true[in_bin])
            bin_positive_rates.append(round(observed_rate, 3))
            bin_centers_output.append(round(center, 3))
            bin_counts.append(count_in_bin)

    return bin_centers_output, bin_positive_rates, bin_counts


def plot_roc(name, y_true, y_pred_proba, **kwargs):
    """
    Plots the ROC curve with hit rate vs false alarm rate (% scale).
    
    Parameters:
        name (str): Label for the curve (e.g., model name).
        y_true (array-like): Ground truth binary labels (0 or 1).
        y_pred_proba (array-like): Predicted probabilities (from model).
        **kwargs: Additional plotting keyword arguments (e.g. linestyle, color).
    """
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    
    plt.plot(100 * fpr, 100 * tpr, label=name, linewidth=1.5, **kwargs)
    plt.xlabel('False Alarm Rate [%]')
    plt.ylabel('Hit Rate [%]')
    plt.xlim([-1, 100])
    plt.ylim([0, 105])
    plt.grid(True, linestyle=':', linewidth=0.5)
    plt.legend(loc='lower right')

In [ ]:
auc_train = roc_auc_score(y_train.flatten(), pred_y_train)
auc_test = roc_auc_score(y_test.flatten(), pred_y_test)

In [ ]:
plot_roc(f"Training set (AUC={auc_train:.3f})", y_train.flatten(), pred_y_train.flatten(), color="red")
plot_roc(f"Testing set  (AUC={auc_test:.3f})", y_test.flatten(), pred_y_test.flatten(), linestyle='--')
plt.legend(loc='lower right');

### Reliability diagram

In [ ]:
prob_pred, prob_true, no_pred_per_bin = reliability_curve(y_test.flatten(), pred_y_test.flatten())

plt.figure(figsize=(5,4))
plt.plot(prob_pred, prob_true, label='Hello')

plt.xlabel("Predicted probability")
plt.ylabel("Obs. proportion of positive case")

no_pred_per_bin = [i/(1.1*np.max(no_pred_per_bin)) for i in no_pred_per_bin]
plt.plot(np.arange(0,1.2,0.2), np.arange(0,1.2,0.2))
plt.bar(prob_pred, no_pred_per_bin, width=0.1, linestyle="--", fill=False, edgecolor="green")

plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='upper center')
plt.show()